# MIDI to Piano-Roll
Loads files listed in `lmd_full_metadata.csv` and converts each MIDI to a piano-roll for CNN training.

Outputs:
- `spectrograms/piano_roll/` — uint8 `.npz` files, shape `(128, T)`, velocity 0–127
- `spectrograms/piano_roll_reduced/` — float16 `.npz` files, shape `(32, T)`, after TruncatedSVD
- `spectrograms/pitch_svd.joblib` — fitted SVD model

Load convention: `roll.astype(np.float32) / 127.0`

In [1]:
import numpy as np
import pandas as pd
import pretty_midi
import os
from pathlib import Path
from tqdm.auto import tqdm

## 1. Load metadata

In [2]:
SCRIPTS_DIR = Path(os.path.abspath(''))
PROJECT_DIR = SCRIPTS_DIR.parent
METADATA_CSV = PROJECT_DIR / 'data' / 'processed_data' / 'lmd_full_metadata.csv'
RAW_DATA_DIR = PROJECT_DIR / 'data' / 'raw_data' / 'lmd_full'
OUT_DIR      = PROJECT_DIR / 'data' / 'processed_data' / 'spectrograms'

OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(METADATA_CSV, parse_dates=False)
print(f"Loaded {len(df):,} files from metadata")
df.head(3)

Loaded 155,681 files from metadata


,file,filepath,duration_s,resolution,n_instruments,n_notes,n_tempo_changes,initial_bpm,n_key_changes,initial_key,n_time_sig_changes,initial_time_sig,key_name
0,d66652ff61cdea1eda265490600e6e40.mid,../data/raw_data/lmd_full/d/d66652ff61cdea1eda...,176.59,384,11,5244,1,118.0,0,NaN,1,4/4,NaN
1,d1e757857fb459edf3e543c3c8a79c6a.mid,../data/raw_data/lmd_full/d/d1e757857fb459edf3...,219.15,192,10,4974,1,72.0,0,NaN,1,4/4,NaN
2,d597590e6423c93180f56f2854bfb109.mid,../data/raw_data/lmd_full/d/d597590e6423c93180...,176.97,384,5,3965,62,120.0,13,0.0,2,1/4,C major


## 2. Configuration

In [3]:
PIANO_ROLL_FS = 100      # frames per second (time resolution)

# Set N_SAMPLES=None to process ALL files.
N_SAMPLES     = None     # process all 155,681 files

# Number of parallel worker processes (CPU-bound). Reduce if you hit memory pressure.
N_WORKERS     = os.cpu_count()

## 3. Conversion helpers

In [4]:
def midi_to_piano_roll(filepath: str, fs: int = PIANO_ROLL_FS) -> np.ndarray:
    """Return a (128, T) piano-roll array (uint8, velocity 0–127).

    Normalize at load time: roll.astype(np.float32) / 127.0
    """
    midi = pretty_midi.PrettyMIDI(filepath)
    roll = midi.get_piano_roll(fs=fs)   # (128, T), velocity 0-127
    return roll.astype(np.uint8)

## 4. Batch conversion and saving

In [5]:
import warnings

def _process_file(args):
    """Worker: parse one MIDI and save its piano-roll as a .npz file.

    Uses fork-based multiprocessing, so all globals (PIANO_ROLL_FS, etc.)
    are inherited from the parent process — no re-import needed.
    Returns None on success, or a dict {'file': ..., 'error': ...} on failure.

    Storage format: uint8 (raw MIDI velocity 0-127) + zlib compression.
    Piano rolls are sparse (~5-10% non-zero), so compression is very effective.
    Normalize to float32 at load time: roll.astype(np.float32) / 127.0
    """
    fpath, pr_out_str = args
    pr_out = Path(pr_out_str)
    if pr_out.exists():
        return None  # idempotent — skip already-processed files
    try:
        # Suppress pretty_midi's RuntimeWarning about non-standard MIDI files
        # (Type 0/1 violations). The files are still parseable; the warning
        # is cosmetic noise that floods output when running in parallel.
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            midi = pretty_midi.PrettyMIDI(fpath)
        roll = midi.get_piano_roll(fs=PIANO_ROLL_FS)   # (128, T), velocity 0-127
        roll = roll.astype(np.uint8)                    # 1 byte/element vs 4 for float32
        np.savez_compressed(pr_out, roll=roll)          # zlib on sparse array ~5-10x smaller
        return None
    except Exception as e:
        return {'file': Path(fpath).name, 'error': str(e)}


In [6]:
import multiprocessing as mp

# ── 1. Sample / duration filter ───────────────────────────────────────────────
df_batch = df.head(N_SAMPLES).reset_index(drop=True) if N_SAMPLES else df.reset_index(drop=True)

MAX_DURATION_S = 300
before = len(df_batch)
df_batch = df_batch[df_batch['duration_s'] <= MAX_DURATION_S].reset_index(drop=True)
print(f"Duration filter (≤ {MAX_DURATION_S}s): {before:,} → {len(df_batch):,} files "
      f"({before - len(df_batch):,} removed)")

piano_roll_dir = OUT_DIR / 'piano_roll'
piano_roll_dir.mkdir(exist_ok=True)

# ── 2. Pre-filter already-processed files (main process, O(1) set lookup) ─────
# Avoids dispatching tasks whose output already exists — no wasted IPC round-trips.
existing     = {p.stem for p in piano_roll_dir.glob('*.npz')}
stems_series = df_batch['file'].str[:-4]           # strip '.mid' suffix — vectorized
todo_mask    = ~stems_series.isin(existing)
n_skip       = int((~todo_mask).sum())
if n_skip:
    print(f"Skipping {n_skip:,} already-processed files.")
df_todo = df_batch[todo_mask].reset_index(drop=True)

# ── 3. Build task list — vectorized (no iterrows) ─────────────────────────────
in_paths  = df_todo['filepath'].str.lstrip('../').apply(lambda p: str(PROJECT_DIR / p)).values
stems_arr = df_todo['file'].str[:-4].values
out_paths = np.array([str(piano_roll_dir / f"{s}.npz") for s in stems_arr])

# ── 4. Sort longest-first for load balancing ──────────────────────────────────
# Distributes heavy files across workers early; prevents a single long file
# from becoming a straggler that blocks the final progress.
sort_idx = df_todo['duration_s'].values.argsort()[::-1]
tasks = list(zip(in_paths[sort_idx], out_paths[sort_idx]))

print(f"Processing {len(tasks):,} files with {N_WORKERS} workers...")

# ── 5. Parallel pool ──────────────────────────────────────────────────────────
# Chunksize: batches IPC messages to reduce coordination overhead.
# Cap at 50 so tqdm ticks frequently enough to show real progress
# (~20 ms/file × 50 = ~1 s between bar updates per worker).
chunksize = max(1, min(50, len(tasks) // (N_WORKERS * 20))) if tasks else 1

errors = []
ctx = mp.get_context('fork')
# maxtasksperchild recycles workers every N tasks to prevent memory creep
# from accumulated pretty_midi C++ objects across thousands of files.
with ctx.Pool(processes=N_WORKERS, maxtasksperchild=1000) as pool:
    for result in tqdm(
        pool.imap_unordered(_process_file, tasks, chunksize=chunksize),
        total=len(tasks),
    ):
        if result is not None:
            errors.append(result)

print(f"\nDone. Errors: {len(errors)}")
if errors:
    err_path = OUT_DIR / 'conversion_errors.csv'
    pd.DataFrame(errors).to_csv(err_path, index=False)
    print(f"Error log saved to {err_path}")

Duration filter (≤ 300s): 155,681 → 139,661 files (16,020 removed)
Skipping 139,661 already-processed files.
Processing 0 files with 14 workers...


0it [00:00, ?it/s]


Done. Errors: 0


## 5. Dimensionality reduction — TruncatedSVD on pitch axis

Piano rolls have 128 pitch bins but most music uses a small subset of pitches — the pitch axis is highly correlated and low-rank.  
We fit a **TruncatedSVD** (sklearn) on a sample of time frames `(N_frames, 128)` to learn the dominant pitch components, then project every saved piano roll from `(128, T)` → `(N_COMPONENTS, T)`.  
The reduced arrays are stored as `float16` `.npz` files in `piano_roll_reduced/`.

> **Why TruncatedSVD and not PCA?**  Piano rolls are *non-negative and sparse*; PCA centers the data (subtracts the mean), which destroys sparsity and makes the result denser.  TruncatedSVD skips centering, works efficiently on sparse inputs, and is numerically stable for this use case.

In [7]:
from sklearn.decomposition import TruncatedSVD
import joblib

# ── TruncatedSVD settings ─────────────────────────────────────────────────────
N_COMPONENTS     = 32        # target pitch dimension (was 128)
SVD_SAMPLE_FILES = 2000      # files sampled to fit the SVD (covers enough variance)
SVD_RANDOM_STATE = 42

REDUCED_DIR    = OUT_DIR / 'piano_roll_reduced'
SVD_MODEL_PATH = OUT_DIR / 'pitch_svd.joblib'
REDUCED_DIR.mkdir(exist_ok=True)

In [8]:
# ── Fit TruncatedSVD on a random sample of time frames ───────────────────────
saved_rolls = sorted(piano_roll_dir.glob('*.npz'))
print(f"Piano-roll files saved: {len(saved_rolls):,}")

rng          = np.random.default_rng(SVD_RANDOM_STATE)
sample_paths = rng.choice(
    saved_rolls,
    size=min(SVD_SAMPLE_FILES, len(saved_rolls)),
    replace=False,
)

print(f'Collecting frames from {len(sample_paths):,} piano rolls...')
frames_list = []
for p in tqdm(sample_paths, desc='loading'):
    roll = np.load(p)['roll'].astype(np.float32) / 127.0  # (128, T)
    frames_list.append(roll.T)                             # (T, 128)

X_fit = np.vstack(frames_list)   # (total_frames, 128)
del frames_list                  # free memory before fitting
print(f'Fitting TruncatedSVD({N_COMPONENTS}) on {X_fit.shape[0]:,} frames × {X_fit.shape[1]} bins...')

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=SVD_RANDOM_STATE, n_iter=5)
svd.fit(X_fit)
del X_fit

joblib.dump(svd, SVD_MODEL_PATH)
print(f'SVD model saved → {SVD_MODEL_PATH}')
cum_var = svd.explained_variance_ratio_.cumsum()[-1]
print(f'Cumulative explained variance ({N_COMPONENTS} components): {cum_var:.4f}')

Piano-roll files saved: 139,661


loading:   0%|          | 0/2000 [00:00<?, ?it/s]

Fitting TruncatedSVD(32) on 34,030,671 frames × 128 bins...
SVD model saved → /Users/yhkim/GU_work/02_Spring_2026/6600_work/04_PROJ/6600_Project/data/processed_data/spectrograms/pitch_svd.joblib
Cumulative explained variance (32 components): 0.7665


In [9]:
# ── Batch transform: (128, T) → (N_COMPONENTS, T), saved as float16 .npz ────
existing_reduced = {p.stem for p in REDUCED_DIR.glob('*.npz')}
tasks_svd = [
    (str(p), str(REDUCED_DIR / p.name))
    for p in sorted(piano_roll_dir.glob('*.npz'))
    if p.stem not in existing_reduced
]
print(f'Transforming {len(tasks_svd):,} files (skipping {len(existing_reduced):,} already done)...')

svd_errors = []
for in_str, out_str in tqdm(tasks_svd, desc='reducing'):
    try:
        roll    = np.load(in_str)['roll'].astype(np.float32) / 127.0  # (128, T)
        reduced = svd.transform(roll.T).T.astype(np.float16)          # (N_COMPONENTS, T)
        np.savez_compressed(out_str, roll=reduced)
    except Exception as e:
        svd_errors.append({'file': in_str, 'error': str(e)})

print(f'Done. Errors: {len(svd_errors)}')
if svd_errors:
    pd.DataFrame(svd_errors).to_csv(OUT_DIR / 'svd_errors.csv', index=False)

Transforming 139,661 files (skipping 0 already done)...


reducing:   0%|          | 0/139661 [00:00<?, ?it/s]

Done. Errors: 0
